# Task 1 — Preparazione del dataset (Bank Marketing)

> 📄 Documentazione completa e motivazioni: [`docs/task1.md`](../docs/task1.md)

## 0. Import e riproducibilità

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
SEED = 10
np.random.seed(SEED)

## 1. Caricamento

In [ ]:
PATH = "../data/raw/bank-additional-full.csv"
df = pd.read_csv(PATH, sep=";")

In [ ]:
print(f"Caricato dataset: {df.shape[0]} istanze x {df.shape[1]} attributi")

In [ ]:
df.head()

## 2. Controllo qualità di base
### 2a. Duplicati

In [ ]:
n_dup = df.duplicated().sum()
print(f"Istanze duplicate trovate: {n_dup}")

In [ ]:
df = df.drop_duplicates().reset_index(drop=True)
print(f"Dataset dopo la rimozione: {df.shape[0]} istanze")

### 2b. Distribuzione della classe

In [ ]:
df["y"].value_counts()

In [ ]:
perc = round(df["y"].value_counts(normalize=True)["yes"] * 100, 2)
print(f"Proporzione 'yes': {perc}%")

### 2c. Valori `unknown`

In [ ]:
# Censimento dei valori 'unknown' nelle colonne nominali (Lezione 3).
# Versione vettorizzata: il confronto (df == "unknown") produce una maschera
# booleana e .sum() conta i True colonna per colonna, senza cicli espliciti.
nominali = df.select_dtypes(include=["object", "string"]).columns

unknown_count = (df[nominali] == "unknown").sum()      # conteggio per colonna
unknown_count = unknown_count[unknown_count > 0]        # solo colonne con unknown

riepilogo_unknown = pd.DataFrame({
    "unknown": unknown_count,
    "%": (unknown_count / len(df) * 100).round(1),
})
print(riepilogo_unknown)

## 3. Codifica della classe

In [ ]:
df["y"] = df["y"].map({"no": 0, "yes": 1})
print("Classe y codificata: no->0, yes->1 (yes = classe positiva)")

In [ ]:
df["y"].value_counts()

## 4. Estrazione dei due file
### 4a. `training.csv`

In [ ]:
import os
os.makedirs("../data/processed", exist_ok=True)

In [ ]:
training = df.copy()
training.to_csv("../data/processed/training.csv", index=False)
print(f"Salvato training.csv: {training.shape[0]} istanze x {training.shape[1]} attributi")

### 4b. `manuale.csv`

In [ ]:
feature_manuale = [
    "age", "campaign",                # numerici
    "job", "marital", "education",    # nominali
    "housing", "loan", "contact", "poutcome",
    "y",                              # classe
]

In [ ]:
# 6 istanze 'yes' e 6 'no' -> set bilanciato (12 istanze)
yes_rows = df[df["y"] == 1].sample(n=6, random_state=SEED)
no_rows  = df[df["y"] == 0].sample(n=6, random_state=SEED)

In [ ]:
manuale = pd.concat([yes_rows, no_rows])[feature_manuale]
# Mescoliamo l'ordine cosi' le classi non sono raggruppate
manuale = manuale.sample(frac=1, random_state=SEED).reset_index(drop=True)

In [ ]:
manuale.to_csv("../data/processed/manuale.csv", index=False)
print(f"Salvato manuale.csv: {manuale.shape[0]} istanze x {manuale.shape[1]} attributi")

In [ ]:
manuale

## Riepilogo

Abbiamo prodotto:
- **`training.csv`** — dataset pulito completo, pronto per EDA e addestramento
- **`manuale.csv`** — 12 istanze bilanciate con attributi adatti ai calcoli a mano

**Prossimo passo (Task 2):** definire a mano i due classificatori (Naïve Bayes e
KNN) su `manuale.csv`, illustrare i passi per adattarli ai dati, implementarli in
Python e valutarne le prestazioni.
